In [167]:
import numpy as np
import random as rd
import matplotlib.pyplot as plt
from scipy.stats import norm

In [168]:
reserve_prices = np.arange(51)*5+670
print(reserve_prices)

[670 675 680 685 690 695 700 705 710 715 720 725 730 735 740 745 750 755
 760 765 770 775 780 785 790 795 800 805 810 815 820 825 830 835 840 845
 850 855 860 865 870 875 880 885 890 895 900 905 910 915 920]


In [169]:
def pnl_calculation(bid1, bid2, avg_bid2):
    # Trades on bid1
    mask1 = reserve_prices < bid1
    n1 = np.sum(mask1)
    pnl1 = n1 * (920 - bid1)

    remaining = reserve_prices[~mask1]

    mask2 = remaining < bid2
    n2 = np.sum(mask2)
    
    # Penalty
    if bid2 >= avg_bid2:
        coeff = 1
    else:
        coeff = ((920 - avg_bid2) / (920 - bid2))**3
    
    pnl2 = n2 * (920 - bid2) * coeff
    
    return pnl1 + pnl2

def find_good_param(avg_bid2):
    max_pnl = -1
    best_bids = (0, 0)
    
    for bid1 in range(670, 921):
        for bid2 in range(bid1, 921):
            pnl = pnl_calculation(bid1, bid2, avg_bid2)
            if pnl >= max_pnl:
                max_pnl = pnl
                best_bids = (bid1, bid2)
                
    return best_bids, max_pnl

In [170]:
print(find_good_param(0))

((751, 836), np.int64(4301))


Pour avg_b2 très inférieur à b2 on a $ optimalbid = (751, 836) $ et cela restera vrai pour $ avg_b2 \in [0, 836]$. Pour un avg_b2 plus haut, on aura bid1 et bid2 optimaux plus hauts. Aussi, il vaut mieux être plus haut que notre approximation de avg_b2 pour ne pas se faire pénaliser. La borne inférieure de 836 pour bid2 indique que le avg_b2 sera compris dans $[836,920]$. 

In [171]:
print(find_good_param(920))

((791, 920), np.int64(3225))


In [172]:
pnl_calculation(791,920,920)

np.int64(3225)

Donc le max qu'on doit mettre pour bid1 est $791$. On serait assuré de gagner 3225 !!

On va regarder pour chaque stratégie optimale à avg_b2 à quel pount il faut avoir sous-estimé avg_b2 pour perdre contre cette stratégie simple

In [173]:

avg_b2_list = []
bid1_list = []
bid2_list = []
pnl_list = []
for avg_b2 in range(836,921):
    (bid1,bid2), pnl = find_good_param(avg_b2)
    avg_b2_list.append(avg_b2)
    bid1_list.append(bid1)
    bid2_list.append(bid2)
    pnl_list.append(pnl)

In [174]:
pnl_calculation(bid1_list[0],bid2_list[0],avg_b2_list[0]+31)

np.float64(3231.68891723356)

In [175]:
pnl_calculation(bid1_list[0],bid2_list[0],avg_b2_list[0]+32)

np.float64(3211.7664399092973)

Ici on voit que pour le plus petit avg_b2 considéré on perd contre la stratégie naïve si on sous estime de 32 points avg_b2.

Calculons l'erreur max autorisé pour les autres valeurs

In [176]:
errors =[]

for i in range(len(avg_b2_list)):
    bid1 = bid1_list[i]
    bid2 = bid2_list[i]
    avg = avg_b2_list[i]
    n=0
    while pnl_calculation(bid1,bid2,avg+n)>3225:
        n+=1
    if n == 0:
        errors.append(n)
    else:
        errors.append(n-1)
print(errors)

[31, 36, 35, 34, 33, 32, 34, 33, 32, 31, 30, 29, 34, 33, 32, 31, 30, 31, 30, 29, 28, 28, 32, 31, 30, 29, 28, 27, 28, 27, 26, 25, 25, 28, 27, 26, 25, 25, 25, 24, 23, 22, 21, 21, 23, 22, 22, 21, 20, 20, 19, 18, 18, 17, 19, 18, 18, 17, 16, 16, 15, 14, 13, 12, 12, 13, 12, 12, 11, 10, 9, 9, 8, 7, 6, 8, 7, 6, 5, 4, 3, 2, 1, 0, 0]


In [177]:
max_avg_b2_val = np.array(avg_b2_list)+np.array(errors)
print(max_avg_b2_val)

[867 873 873 873 873 873 876 876 876 876 876 876 882 882 882 882 882 884
 884 884 884 885 890 890 890 890 890 890 892 892 892 892 893 897 897 897
 897 898 899 899 899 899 899 900 903 903 904 904 904 905 905 905 906 906
 909 909 910 910 910 911 911 911 911 911 912 914 914 915 915 915 915 916
 916 916 916 919 919 919 919 919 919 919 919 919 920]


On voit que certaine valeurs de avg_b2 partagent le même max à ne pas dépasser donc si c'est pour choisir deux dont le max avg_b2 à ne pas dépasser est identitique, autant prendre le plus petit car il permet de gagner plus potentiellement que la stratégie simple.

In [178]:
unique_most_left_index = [0]
val = max_avg_b2_val[0]
for index in range(1,len(max_avg_b2_val)):
    if max_avg_b2_val[index]!= val:
        unique_most_left_index.append(index)
        val = max_avg_b2_val[index]

print(unique_most_left_index)

[0, 1, 6, 12, 17, 21, 22, 28, 32, 33, 37, 38, 43, 44, 46, 49, 52, 54, 56, 59, 64, 65, 67, 71, 75, 84]


In [179]:
avg_b2_list = np.array(avg_b2_list)[unique_most_left_index]
bid1_list = np.array(bid1_list)[unique_most_left_index]
bid2_list = np.array(bid2_list)[unique_most_left_index]
pnl_list = np.array(pnl_list)[unique_most_left_index]
errors = np.array(errors)[unique_most_left_index]

On a éliminé des valeurs maintenant cherchont les valeurs de avg_b2 qui maximisent un "sharp ratio" customisé

On va faire $S_a = \frac{pnl(avgb2)-3225}{error(avgb2)}$

In [184]:
errors = np.where(errors ==0, 1,errors)
sharpe_ratio = (pnl_list - 3225)/errors

In [185]:
print(pnl_list)

[4301 4295 4284 4263 4237 4218 4201 4160 4120 4109 4069 4053 3990 3987
 3966 3916 3872 3835 3813 3749 3657 3653 3607 3528 3441 3225]


In [186]:
print(sharpe_ratio)

[34.70967742 29.72222222 31.14705882 30.52941176 32.64516129 35.46428571
 30.5        33.39285714 35.8        31.57142857 33.76       33.12
 36.42857143 33.13043478 33.68181818 34.55       35.94444444 32.10526316
 32.66666667 32.75       36.         32.92307692 31.83333333 33.66666667
 27.          0.        ]


In [189]:
good_val_index = sharpe_ratio >=35

max_avg_b2_val = max_avg_b2_val[unique_most_left_index]

print(avg_b2_list[good_val_index], pnl_list[good_val_index], max_avg_b2_val[good_val_index])

[857 868 879 888 900] [4218 4120 3990 3872 3657] [885 893 900 906 912]


A partir de là on pourrait choisir n'importe quelle valeur, mais on va plutôt prendre celle avec le meilleur sharpe ratio pour ne pas trop réfléchir davantage.

In [190]:
ans_index = np.argmax(sharpe_ratio)

print(avg_b2_list[ans_index],pnl_list[ans_index], sharpe_ratio[ans_index])

879 3990 36.42857142857143


In [191]:
print(find_good_param(879))

((771, 879), np.int64(3990))


Résultat final: bid1 , bid2 = 771, 879 

Gain maximal = 3990

Pire avg_b2 à ne pas dépasser = 900